1. Imports & Project Paths

In [1]:
from pathlib import Path
import json
from typing import cast
from zoneinfo import available_timezones

import numpy as np
import pandas as pd
import joblib
import requests
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = PROCESSED_DIR / "products_cleaned.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())
print("Models directory:", MODELS_DIR)

/Users/macbookpro/PycharmProjects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/macbookpro/PycharmProjects/Techcart_ai
Dataset exists: True
Models directory: /Users/macbookpro/PycharmProjects/Techcart_ai/models


2. Load and inspect data

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} မတွေ့ပါ။ Notebook 1 ကို အရင် run ပါ။"
    )

df = pd.read_csv(DATA_PATH)
REQUIRED_COLUMNS = [
    "product_id",
    "name",
    "category",
    "brand",
    "price",
    "stock",
    "rating"
]

missing_columns = [
    col for col in REQUIRED_COLUMNS if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

df = df[REQUIRED_COLUMNS].copy()
df=df.dropna(
    subset=["product_id", "name", "category"]).copy()

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (196, 7)


,product_id,name,category,brand,price,stock,rating
0,1,MacBook Pro M4 14,Laptop,Apple,10677.59,10.0,4.7
1,2,RTX 5070 Ti 16GB,GPU,NVIDIA,771.27,13.0,4.9
2,3,Core Ultra 5 245K,CPU,Intel,323.46,12.0,4.3
3,4,16GB DDR5-5600,RAM,Kingston,50.02,47.0,4.2
4,5,27in 4K OLED 240Hz,Monitor,ASUS,891.30,5.0,4.7


3. Create documents

In [3]:
def safe_value(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip()

    if value == "":
        return "Unknown"

    return value


def product_to_document(row):
    return(
        f"Product ID: {safe_value(row['product_id'])} \n"
        f"Product Name: {safe_value(row['name'])} \n"
        f"Category: {safe_value(row['category'])} \n"
        f"Brand: {safe_value(row['brand'])} \n"
        f"Price: {safe_value(row['price'])} \n"
        f"Stock: {safe_value(row['stock'])} \n"
        f"Rating: {safe_value(row['rating'])} \n"
    )

documents = [
    product_to_document(row)
    for _, row in df.iterrows()
]

metadata = df.to_dict(orient="records")

print("Number of product documents:", len(documents))
print("\nExample of documents:")
print(documents[0])

Number of product documents: 196

Example of documents:
Product ID: 1 
Product Name: MacBook Pro M4 14 
Category: Laptop 
Brand: Apple 
Price: 10677.59 
Stock: 10.0 
Rating: 4.7 



4. Load embedding model

In [4]:
EMBEDDING_MODEL_NAME = ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5176.10it/s]


Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


5. Generate document embeddings

In [5]:
document_embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

document_embeddings = np.asarray(
    document_embeddings,
    dtype=np.float32
)

print("Embedding Shape:", document_embeddings.shape)
print("Number of documents:", len(documents))

Batches: 100%|██████████| 7/7 [00:00<00:00, 10.29it/s]

Embedding Shape: (196, 384)
Number of documents: 196


6. Save vector store

In [6]:
VECTOR_STORE_PATH = (MODELS_DIR / "product_catalog_vector_store.joblib")

vector_store = {"documents": documents,
                "metadata": metadata,
                "embeddings": document_embeddings,
                "embedding_model_name": EMBEDDING_MODEL_NAME}

joblib.dump(vector_store, VECTOR_STORE_PATH)

print("Vector store saved:", VECTOR_STORE_PATH)
print("File exists:", VECTOR_STORE_PATH.exists())

Vector store saved: /Users/macbookpro/PycharmProjects/Techcart_ai/models/product_catalog_vector_store.joblib
File exists: True


7. Retrieve relevant products

In [7]:
def retrieve_products(query, top_k=4):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query u စာသားလွတ် မဖြစ်ရပါ")

    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.asarray(query_embedding, dtype=np.float32)
    scores = (vector_store["embeddings"] @ query_embedding)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "score": float(scores[index]),
            "document": vector_store["documents"][index],
            "metadata": vector_store["metadata"][index]
        })

    return results

8. Test Retrieval

In [8]:
query = "Samsung SSD products"

results = retrieve_products(query, top_k=4)

for i, results in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Similarity:", round(results["score"], 4))
    print(results["document"])


--- Result 1 ---
Similarity: 0.6584
Product ID: 19 
Product Name: 2TB 990 PRO 
Category: SSD 
Brand: Samsung 
Price: 161.14 
Stock: 30.0 
Rating: 4.9 


--- Result 2 ---
Similarity: 0.654
Product ID: 187 
Product Name: 2TB 990 PRO 
Category: SSD 
Brand: Samsung 
Price: 176.65 
Stock: 16.0 
Rating: 4.7 


--- Result 3 ---
Similarity: 0.6492
Product ID: 153 
Product Name: 2TB 990 PRO 
Category: SSD 
Brand: Samsung 
Price: 172.35 
Stock: 30.0 
Rating: 4.9 


--- Result 4 ---
Similarity: 0.6477
Product ID: 7 
Product Name: 2TB 990 PRO 
Category: SSD 
Brand: Samsung 
Price: 162.96 
Stock: 31.0 
Rating: 4.8 



9. Check Ollama

In [9]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:3b"

try:
    response = requests.get(
        "http://localhost:11434/api/tags", timeout=5)

    response.raise_for_status()

    available_models = [item["name"] for item in response.json().get("models",[])]

    print("Ollama is Running...")
    print("Available models:", available_models)

    if not any(
        name.split(":")[0] ==  OLLAMA_MODEL.split(":")[0]
        for name in available_models
    ):
        print(f"Model {OLLAMA_MODEL} not available"
              f"Terminal မှာ 'ollama pull {OLLAMA_MODEL}' run ပါ")


except requests.RequestException as error:
    print("Ollama ကို မချိတ်ဆတ်နိုင်ပါ")
    print("Ollama application ကို ဖွင့်ထားကြောင်း စစ်ပါ")
    print("Error:", error)

Ollama is Running...
Available models: ['qwen2.5:3b']


10. Build RAG answer function

In [10]:
def ask_product_rag(question, top_k=2):

    # 1. Retrieve fewer documents to reduce prompt size
    retrieved = retrieve_products(
        question,
        top_k=top_k
    )

    context = "\n\n".join(
        item["document"]
        for item in retrieved
    )

    # 2. Use a shorter prompt
    prompt = f"""
You are TechCart AI, a product assistant.

Answer briefly using only the context below.
Do not invent prices or stock values.
If information is missing, say so.
Answer in Burmese if the question is in Burmese.

Context:
{context}

Question:
{question}

Answer:
"""

    # 3. Generate with a shorter output limit
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "keep_alive": "10m",
            "options": {
                "temperature": 0,
                "num_predict": 128
            }
        },
        timeout=(10, 600)
    )

    response.raise_for_status()

    answer = response.json().get(
        "response", ""
    ).strip()

    return {
        "question": question,
        "answer": answer,
        "retrieved_products": retrieved
    }

11. Ask a question

In [11]:
result = ask_product_rag(
    "Samsung SSD products ရဲ့ price နဲ့ stock ကို ပြပေးပါ",
    top_k=2
)

print("RAG Answer:")
print(result["answer"])

print("\nRetrieved sources:")
for item in result["retrieved_products"]:
    print(item["document"])
    print()

RAG Answer:
Samsung SSD အတိုင်းရဲ့ price: 161.14 မျိုး
stock: 30.0 ကုဒ်

Retrieved sources:
Product ID: 187 
Product Name: 2TB 990 PRO 
Category: SSD 
Brand: Samsung 
Price: 176.65 
Stock: 16.0 
Rating: 4.7 


Product ID: 19 
Product Name: 2TB 990 PRO 
Category: SSD 
Brand: Samsung 
Price: 161.14 
Stock: 30.0 
Rating: 4.9 




12. Save Metadata

In [12]:
RAG_METADATA_PATH = MODELS_DIR / "product_rag_metadata.json"

rag_metadata = {
    "project": "TechCart AI",
    "method": "Retrieval-Augmented Generation",
    "embedding_model": EMBEDDING_MODEL_NAME,
    "llm_model": OLLAMA_MODEL,
    "document_count": len(documents),
    "vector_dimension": int(document_embeddings.shape[1]),
    "vector_store": str(VECTOR_STORE_PATH.name)
}

with open(
    RAG_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        rag_metadata,
        file,
        indent=2,
        ensure_ascii=False
    )

print("RAG metadata saved:", RAG_METADATA_PATH)

RAG metadata saved: /Users/macbookpro/PycharmProjects/Techcart_ai/models/product_rag_metadata.json


In [13]:
def show_product_results(query, top_k=4):

    results = retrieve_products(
        query,
        top_k=top_k
    )

    rows = []

    for result in results:
        product = result["metadata"]

        rows.append({
            "Product ID": product.get("product_id"),
            "Product Name": product.get("name"),
            "Brand": product.get("brand"),
            "Category": product.get("category"),
            "Price": product.get("price"),
            "Stock": product.get("stock"),
            "Rating": product.get("rating"),
            "Similarity": round(result["score"], 4)
        })

    results_df = pd.DataFrame(rows)

    display(results_df)

    return results_df

product_results = show_product_results(
    "Samsung SSD products",
    top_k=4
)

,Product ID,Product Name,Brand,Category,Price,Stock,Rating,Similarity
0,19,2TB 990 PRO,Samsung,SSD,161.14,30.0,4.9,0.6584
1,187,2TB 990 PRO,Samsung,SSD,176.65,16.0,4.7,0.6540
2,153,2TB 990 PRO,Samsung,SSD,172.35,30.0,4.9,0.6492
3,7,2TB 990 PRO,Samsung,SSD,162.96,31.0,4.8,0.6477
